# Occupancy Optimization

## Stencil Code Optimization 2 - Increase Number of Blocks

To optimize performance, we need to increase the number of blocks.
One first attempt could be reducing the block size.
OpenMP supports this by specifying the `num_threads` clause.
The updated code is available at [stencil-2d-omp-target-v2](../src/stencil-2d/stencil-2d-omp-target-v2.cpp), and can be compiled, executed and profiled with the following cells.

In [ ]:
!nvc++ -O3 -march=native -std=c++17 -mp=gpu -target=gpu ../src/stencil-2d/stencil-2d-omp-target-v2.cpp -o ../build/stencil-2d-omp-target-v2

In [ ]:
!../build/stencil-2d-omp-target-v2 double 8192 8192 2 256

In [ ]:
!ncu -s 2 -c 1 ../build/stencil-2d-omp-target-v2 double 8192 8192 2 2

### Potential Output

```
  nvkernel__Z9stencil2dIdEvPKT_PS0_mm_F1L5_6 (128, 1, 1)x(64, 1, 1), Context 1, Stream 13, Device 0, CC 8.6
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         7.24
    SM Frequency                    Ghz         1.30
    Elapsed Cycles                cycle    9,326,380
    Memory Throughput                 %        42.95
    DRAM Throughput                   %        21.73
    Duration                         ms         7.15
    L1/TEX Cache Throughput           %        51.46
    L2 Cache Throughput               %        35.93
    SM Active Cycles              cycle 7,780,157.35
    Compute (SM) Throughput           %        17.14
    ----------------------- ----------- ------------

    OPT   This kernel grid is too small to fill the available resources on this device, resulting in only 0.10 full     
          waves across all SMs. Look at Launch Statistics for more details.                                             
```

```
    Section: Launch Statistics
    -------------------------------- --------------- ---------------
    Metric Name                          Metric Unit    Metric Value
    -------------------------------- --------------- ---------------
    Block Size                                                    64
    Function Cache Configuration                     CachePreferNone
    Grid Size                                                    128
    Registers Per Thread             register/thread              36
    Shared Memory Configuration Size           Kbyte           16.38
    Driver Shared Memory Per Block       Kbyte/block            1.02
    Dynamic Shared Memory Per Block       byte/block               0
    Static Shared Memory Per Block        byte/block               0
    # SMs                                         SM              84
    Stack Size                                                 1,024
    Threads                                   thread           8,192
    # TPCs                                                        42
    Enabled TPC IDs                                              all
    Uses Green Context                                             0
    Waves Per SM                                                0.10
    -------------------------------- --------------- ---------------

    OPT   If you execute __syncthreads() to synchronize the threads of a block, it is recommended to have at least two  
          blocks per multiprocessor (compared to the currently executed 1.5 blocks) This way, blocks that aren't        
          waiting for __syncthreads() can keep the hardware busy.                                                       
```

```
    Section: Occupancy
    ------------------------------- ----------- ------------
    Metric Name                     Metric Unit Metric Value
    ------------------------------- ----------- ------------
    Block Limit SM                        block           16
    Block Limit Registers                 block           24
    Block Limit Shared Mem                block           16
    Block Limit Warps                     block           24
    Theoretical Active Warps per SM        warp           32
    Theoretical Occupancy                     %        66.67
    Achieved Occupancy                        %         6.71
    Achieved Active Warps Per SM           warp         3.22
    ------------------------------- ----------- ------------

    OPT   Est. Local Speedup: 89.93%                                                                                    
          The difference between calculated theoretical (66.7%) and measured achieved occupancy (6.7%) can be the       
          result of warp scheduling overheads or workload imbalances during the kernel execution. Load imbalances can   
          occur between warps within a block as well as across blocks of the same kernel. See the CUDA Best Practices   
          Guide (https://docs.nvidia.com/cuda/cuda-c-best-practices-guide/index.html#occupancy) for more details on     
          optimizing occupancy.                                                                                         
    ----- --------------------------------------------------------------------------------------------------------------
    OPT   Est. Local Speedup: 33.33%                                                                                    
          The 8.00 theoretical warps per scheduler this kernel can issue according to its occupancy are below the       
          hardware maximum of 12. This kernel's theoretical occupancy (66.7%) is limited by the number of blocks that   
          can fit on the SM, and the required amount of shared memory.                                                  
```

```
    Section: GPU and Memory Workload Distribution
    -------------------------- ----------- -------------
    Metric Name                Metric Unit  Metric Value
    -------------------------- ----------- -------------
    Average DRAM Active Cycles       cycle 11,250,685.33
    Total DRAM Elapsed Cycles        cycle   621,225,984
    Average L1 Active Cycles         cycle  7,780,157.35
    Total L1 Elapsed Cycles          cycle   782,925,996
    Average L2 Active Cycles         cycle  8,768,759.40
    Total L2 Elapsed Cycles          cycle   421,939,824
    Average SM Active Cycles         cycle  7,780,157.35
    Total SM Elapsed Cycles          cycle   782,925,996
    Average SMSP Active Cycles       cycle  6,265,809.45
    Total SMSP Elapsed Cycles        cycle 3,131,703,984
    -------------------------- ----------- -------------

    OPT   Est. Speedup: 13.84%                                                                                          
          One or more SMs have a much lower number of active cycles than the average number of active cycles. Maximum   
          instance value is 16.58% above the average, while the minimum instance value is 19.59% below the average.     
    ----- --------------------------------------------------------------------------------------------------------------
    OPT   Est. Speedup: 22.05%                                                                                          
          One or more SMSPs have a much lower number of active cycles than the average number of active cycles. Maximum 
          instance value is 32.80% above the average, while the minimum instance value is 100.00% below the average.    
    ----- --------------------------------------------------------------------------------------------------------------
    OPT   Est. Speedup: 13.84%                                                                                          
          One or more L1 Slices have a much lower number of active cycles than the average number of active cycles.     
          Maximum instance value is 16.58% above the average, while the minimum instance value is 19.59% below the      
          average.                                                                                                      
```

### Interpretation

While our application now utilized all SMs, the other part of the previous problem - not enough (full) waves and a low occupancy - has not been fixed.
As such, the performance is still comparatively low as well.
At this point you might ask yourself: what is a *wave*?

To answer this, let's clarify how threads are organized on GPUs:
- **OpenMP** organizes *threads* in *teams*.
- **CUDA** organizes *threads* in *blocks*.

Although OpenMP teams and CUDA thread blocks are not strictly equivalent, compilers typically map teams to blocks when targeting NVIDIA GPUs.
These blocks are then assigned to Streaming Multiprocessors (SMs):
* Each block is scheduled to one SM.
* Each SM can handle multiple blocks concurrently (depending on available resources).

Within a block, threads are further grouped into *warps*: groups of 32 threads.
Warps are the basic unit of scheduling on the SM sub-partitions (SMSPs).

A *wave* refers to the maximum number of blocks that can be resident on the GPU at the same time.
It is calculated as the number of SMs times the maximum number of blocks per SM (for a given kernel and execution configuration).

Having at least one full wave ensures that all resources are fully utilized.
Multiple (full) waves work as well, but partial waves may lead to sub-optimal performance.

This closely relates to the concept of *occupancy* which is defined as the ratio of active warps to the maximum number of warps that could be scheduled on an SM.

## Micro Benchmarks

At this point, you might wonder whether achieving full waves is always necessary, or if a lower degree of parallelism can sometimes suffice.

There are two main ways to approach this question:

1. **Conceptual Analysis:** \
To have a GPU fully utilize its computational resources, it requires at least as many active threads as there are execution units for the current data type - typically on the order of ten thousand.
However, most applications require additional data transfers between main memory (or at least L2 cache) and the compute units, which introduces high latency.
To hide this latency and keep the GPU busy, resources need to be oversubscribed: when some warps are stalled waiting for data, others can be scheduled to execute.
In practice this usually relates to an order of magnitude more threads required (on the order of a hundred thousand).

2. **Benchmarking:** \
(Micro) benchmarks can help study performance for varying number of threads.
They can additionally be used to examine the effects of different data types, memory access patterns, computational intensities, the ratio of compute to data transfer, and more.


## Next Step

Head over to the [micro benchmarks](./07-micro-benchmarks.ipynb) notebook to get started.